# softmax回归实现

In [3]:
from IPython import display
import torch
import torchvision
from d2l import torch as d2l

In [4]:


batch_size=256
torchvision.datasets.FashionMNIST.mirrors = [
    'https://raw.githubusercontent.com/zalandoresearch/fashion-mnist/master/data/fashion/'
]
train_iter,test_iter=d2l.load_data_fashion_mnist(batch_size)

num_inputs=28*28
num_outputs=10

W=torch.normal(0,0.01,size=(num_inputs,num_outputs),requires_grad=True)
b=torch.zeros(num_outputs,requires_grad=True)



In [5]:
X=torch.tensor([[1.0,2.0,3.0],[4.0,5.0,6.0]])
# 0 压缩成一行 1 压缩成1列
X.sum(0,keepdim=True),X.sum(1,keepdim=True)


(tensor([[5., 7., 9.]]),
 tensor([[ 6.],
         [15.]]))

In [6]:
# d2l 习惯: 每一行一个样本,行数是样本数,列数是特征数
# X (256,10)
def softmax(X):
    X_exp=torch.exp(X) # 256*10
    X_exp_sum=X_exp.sum(1,keepdim=True) # 256*1
    return X_exp/X_exp_sum

In [7]:
X=torch.normal(0,1,(2,5))
X_prob=softmax(X)
X_prob,X_prob.sum(1)

(tensor([[0.0579, 0.1889, 0.4151, 0.1824, 0.1556],
         [0.2441, 0.0716, 0.0545, 0.5141, 0.1157]]),
 tensor([1., 1.]))

In [8]:
def net(X):
    return softmax(torch.matmul(X.reshape((-1,W.shape[0])),W)+b)

In [9]:
y=torch.tensor([0,2])
y_hat=torch.tensor([[0.1,0.3,0.6],[0.3,0.2,0.5]])
# y_hat[数组下标]
# 表示取y_hat的0,1行,一一对应0,2列
# y_hat[[0,1,2],[1,2,3]]表示把[0,1,2]和[1,2,3]分别配对成(0,1),(1,2),(2,3)
# 然后取出y_hat的这三个坐标的值,y_hat[0][1],y_hat[1][2],y_hat[2][3]
#在把这三个数字放一个数组里面
#神奇的语法 
y_hat[[0,0,0,1,1,1],[0,1,2,0,1,2]] 
# 等价写法  -1就是自己猜, 因为有等式是变形前后元素总数不变
# y_hat.reshape(-1)

tensor([0.1000, 0.3000, 0.6000, 0.3000, 0.2000, 0.5000])

In [10]:
def cross_entropy(y_hat,y):
    return -torch.log(y_hat[range(len(y_hat)),y])
cross_entropy(y_hat,y)



tensor([2.3026, 0.6931])

## 交叉熵损失函数细节

### 1. `cross_entropy` 为什么返回向量而不是标量？

```python
def cross_entropy(y_hat, y):
    return -torch.log(y_hat[range(len(y_hat)), y])
```

这段代码**故意返回每个样本的损失向量** `(batch_size,)`，而不是标量。原因是：
- d2l 的训练循环通常会在外面做 `l.sum().backward()` 或 `l.mean().backward()`
- 保留向量更灵活，方便后续加权、求和或求平均
- 等价于 PyTorch 的 `F.cross_entropy(..., reduction='none')`

如果希望直接返回标量，可以手动加 `.mean()`：

```python
return -torch.log(y_hat[range(len(y_hat)), y]).mean()
```

---

### 2. `range(len(y_hat))` 返回值是 list 吗？

**不是**。Python 3 中 `range()` 返回的是 **`range` 对象**，不是 `list`：

```python
>>> type(range(5))
<class 'range'>

>>> list(range(5))
[0, 1, 2, 3, 4]
```

PyTorch tensor 支持 `range` 对象作为索引，所以可以直接用。更清晰的写法是 `torch.arange(len(y_hat))`。

---

### 3. `len(y_hat)` 等于 `y_hat.shape[0]` 吗？

**是的**，在 PyTorch 中通常相等：

```python
len(y_hat) == y_hat.shape[0]  # True
```

因为 `Tensor.__len__()` 的实现就是返回第一维的大小。

⚠️ 例外：0维张量（标量）不能用 `len()`，会报错 `TypeError: len() of a 0-d tensor`。


In [11]:
def accuracy(y_hat,y):
    if len(y_hat.shape)>1 and y_hat.shape[1]>1:
        y_hat=y_hat.argmax(axis=1)  # 0--行 1--列  拍成列,arg:每行最大概率的对应的索引
    cmp=y_hat.type(y.dtype)==y # bool array
    return float(cmp.type(y.dtype).sum())


## 准确率函数 `accuracy` 细节

### 1. `if len(y_hat.shape) > 1` 的含义

```python
if len(y_hat.shape) > 1 and y_hat.shape[1] > 1:
    y_hat = y_hat.argmax(axis=1)
```

| 情况 | `y_hat` 形状 | 含义 | 处理 |
|------|-------------|------|------|
| 多分类概率 | `(batch, num_classes)` | 每行是各类别概率 | 用 `argmax` 取预测类别 |
| 已预测标签 | `(batch,)` | 直接是类别编号 | 跳过 |

`len(y_hat.shape)` 返回的是**维度数**（ndim），`> 1` 表示至少是 2 维。

---

### 2. `argmax` 的作用

`argmax` 返回**最大值的索引**，在多分类里就是找出概率最高的类别：

```python
y_hat = torch.tensor([[0.1, 0.7, 0.2],
                      [0.8, 0.1, 0.1]])
pred = y_hat.argmax(dim=1)  # tensor([1, 0])
```

- `dim=0`：纵向按列找（结果形状 `(num_classes,)`）
- `dim=1`：横向按行找（结果形状 `(batch_size,)`）

与 `max` 的区别：`max` 返回**最大值本身**，`argmax` 返回**最大值的位置**。

---

### 3. `cmp = y_hat.type(y.dtype) == y` 拆解

```python
cmp = y_hat.type(y.dtype) == y
```

| 部分 | 作用 |
|------|------|
| `y_hat.type(y.dtype)` | 把 `y_hat` 强制转成和 `y` 相同的数据类型 |
| `== y` | 逐元素比较，返回布尔张量 `True/False` |

为什么要转类型？因为 `y_hat`（预测标签）和 `y`（真实标签）的数据类型可能不同（如 `int64` vs `float32`），必须统一才能正确比较。

---

### 4. `.type()` vs `.dtype` 的区别

| | `.dtype` | `.type()` |
|--|---------|-----------|
| **性质** | 属性（只读） | 方法（可转换） |
| **返回** | `torch.int64` | 类型名称字符串 `'torch.LongTensor'` |
| **作用** | **查看**类型 | **转换**类型 |

```python
x.dtype                # torch.int64          ← 查类型
x.type()               # 'torch.LongTensor'   ← 查类型（字符串）
x.type(torch.float32)  # tensor([1., 2.])     ← 转类型
x.to(torch.float32)    # tensor([1., 2.])     ← 更推荐的新写法
```

> `.dtype` 是"你是什么类型"，`.type(...)` 是"我要把你变成什么类型"。

---

### 5. `accuracy` 为什么返回正确个数，而不是 (0,1) 之间的准确率？

```python
return float(cmp.type(y.dtype).sum())  # 返回正确个数
```

因为训练是**分批（batch）**进行的，直接返回比例会导致跨 batch 平均时出错（尤其最后一个 batch 可能不满）。

d2l 的做法是：返回**正确个数**，训练循环里用 `Accumulator` 分别累加**总正确数**和**总样本数**，最后再相除：

```python
metric.add(accuracy(net(X), y), y.numel())
# 最后
acc = metric[0] / metric[1]  # 总正确数 / 总样本数 ✓
```

这样即使最后一个 batch 只有 1 个样本，也不会影响最终准确率。

如果 `accuracy` 直接除以 `len(y)` 返回比例，跨 batch 直接累加平均会得到错误结果。


In [12]:
accuracy(y_hat,y)/len(y)

0.5

In [13]:
from d2l.torch import Accumulator


def evaluate_accuracy(net,data_iter):
    if isinstance(net,torch.nn.Module):
        net.eval() # start evaluation mode 
    # Accumulator 累加器,实际上就是一个破list
    metric=Accumulator(2) # 建一个len=2 list
    with torch.no_grad(): # close gradient to be efficient 如果不加:梯度会被缓存,多个batch会爆内存
        for X,y in data_iter:
    # Accumulator add :list里面的数逐个加add()的参数,因此数量要一致
            metric.add(accuracy(net(X),y),y.numel()) #numel 总元素数 
    return metric[0]/metric[1]


In [14]:
def train_epoch_ch3(net,train_iter,loss,updater):
    if isinstance(updater,torch.nn.Module):
        net.train()# 训练模式
    # cost,good cases,all samples
    metric=Accumulator(3)
    for X,y  in train_iter:
        y_hat=net(X)
        l=loss(y_hat,y)
        if isinstance(updater,torch.optim.Optimizer):
            updater.zero_grad()
            l.mean().backward()
        else:
            l.sum().backward()
            updater(X.shape[0])
        metric.add(float(l.sum()),accuracy(y_hat,y),y.numel())
    
    # mean cost, percentage accuracy
    return metric[0]/metric[2],metric[1]/metric[2]


